In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
docs_llm = [doc for doc in documents if doc['course'] == 'llm-zoomcamp']
len(docs_llm)

118

In [4]:
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    id_field='doc_id',      # <-- use the existing id so if it's re-run, it will update existing records instead of creating duplicates
    db_path='faq.db'
)

In [5]:
import time

for doc in docs_llm:
    doc['doc_id'] = doc.pop('id') # "id" is already a reserved column in sqlite, need to add the ids from the json as a different field name
    index.add(doc)
    print(f'Added: {doc["question"][:60]}...')
    time.sleep(0.5)

Added: I just discovered the course. Can I still join?...
Added: Course: I have registered for the LLM Zoomcamp. When can I e...
Added: What is the video/zoom link to the stream for the “Office Ho...
Added: How should I start the course and follow the weekly workflow...
Added: Leaderboard: I am not on the leaderboard / how do I know whi...
Added: Certificate: Can I follow the course in a self-paced mode an...
Added: I missed the first homework - can I still get a certificate?...
Added: Homework: Why does the content keep changing?...
Added: When will the course be offered next?...
Added: Are there any lectures/videos? Where are they?...
Added: Where can I track the LLM Zoomcamp syllabus, deadlines, home...
Added: Are there live sessions or office hours for each module?...
Added: Can I use Bluesky for learning in public credits?...
Added: Where is the LLM Zoomcamp Telegram channel?...
Added: Why doesn't the number of records I get in the FAQ dataset m...
Added: The homework submission f

### sqlite data

In [33]:
import sqlite3
import pandas as pd

In [ ]:
pd.set_option('display.max_columns', None)  # Show all columns
from IPython.display import display, clear_output

In [39]:
def read_sqlite_db(db_path: str):
    # Connect to the database file
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row  # allows accessing columns by name
    cur = conn.cursor()

    try:
        # Get a list of all user tables
        cur.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
              AND name NOT LIKE 'sqlite_%'
            ORDER BY name
        """)
        tables = [row[0] for row in cur.fetchall()]

        for table in tables:
            print(f"Reading table: {table}, number of rows: {cur.execute(f'SELECT COUNT(*) FROM \"{table}\"').fetchone()[0]}")
            print(f"Columns: {[desc[0] for desc in cur.execute(f'SELECT * FROM \"{table}\"').description]}")

            df =pd.read_sql_query(f'SELECT * FROM "{table}" LIMIT 5', conn).head()  # Display first 5 rows using pandas
            display(df)

            print()

    finally:
        conn.close()

In [40]:
read_sqlite_db('faq.db')

Reading table: docs, number of rows: 118
Columns: ['id', 'doc_json', 'vector_hash', 'course', 'doc_id']


,id,doc_json,vector_hash,course,doc_id
0,1,"{""course"": ""llm-zoomcamp"", ""section"": ""General...",None,llm-zoomcamp,74eb249bbf
1,2,"{""course"": ""llm-zoomcamp"", ""section"": ""General...",None,llm-zoomcamp,977bf7786c
2,3,"{""course"": ""llm-zoomcamp"", ""section"": ""General...",None,llm-zoomcamp,489dd1c9d9
3,4,"{""course"": ""llm-zoomcamp"", ""section"": ""General...",None,llm-zoomcamp,04919992b3
4,5,"{""course"": ""llm-zoomcamp"", ""section"": ""General...",None,llm-zoomcamp,c2903069a0



Reading table: docs_fts, number of rows: 118
Columns: ['docid', 'question', 'section', 'answer']


,docid,question,section,answer
0,1,I just discovered the course. Can I still join?,General Course-Related Questions,"Yes, but if you want to receive a certificate,..."
1,2,Course: I have registered for the LLM Zoomcamp...,General Course-Related Questions,You don't need it. You're accepted. You can al...
2,3,What is the video/zoom link to the stream for ...,General Course-Related Questions,The zoom link is only published to instructors...
3,4,How should I start the course and follow the w...,General Course-Related Questions,Start with the [LLM Zoomcamp docs](https://dat...
4,5,Leaderboard: I am not on the leaderboard / how...,General Course-Related Questions,"When you set up your account, you are automati..."



Reading table: docs_fts_config, number of rows: 1
Columns: ['k', 'v']


...

,id,block
0,1,b'vv\x8dG\x83\x0f\xd8\x1e'
1,10,b'\x00\x00\x00\x00\x03\nv\x00\x06\x02\x01\x01\...
2,137438953473,b'\x00\x00\x0f\x0f\x0200\x1c\x08\x01\x03\x81\x...
3,137438953474,b'\x00\x00\x0f\'\x080awesome\x17\x06\x01\x031\...
4,137438953475,b'\x00\x06\x0f\n\x03C*\x08\x01\x03\x82\n\x07\x...



Reading table: docs_fts_docsize, number of rows: 118
Columns: ['id', 'sz']


,id,sz
0,1,b'\x01\t\x04\x15'
1,2,b'\x01\x11\x04*'
2,3,b'\x01\x11\x04R'
3,4,b'\x01\x0b\x04t'
4,5,b'\x01\x12\x04\x81\x05'



Reading table: docs_fts_idx, number of rows: 25
Columns: ['segid', 'term', 'pgno']


,segid,term,pgno
0,1,b'',2
1,1,b'0awe',4
2,1,b'0date',6
3,1,b'0generat',8
4,1,b'0keepi',10
